# 主动源面波：一步步运行

## Goal

本 Notebook 只做主动源。流程为：**集中参数 → 输入图/required 检查 → 多分量炮集 → Hankel/FJ 频散 → 理论曲线 → 保存**。

日常操作只修改第 2 步的一个 `PARAMETERS` Cell。设置 `input_toml` 后则完全读取 TOML，其余 Cell 参数会被明确忽略。


## Setup

### 1. 导入同一套 Python/C++ 计算核心


In [1]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

from wavedispx.notebook_support import (
    input_error_markdown,
    parameter_reference_markdown,
    prepare_notebook_case,
    required_failure_markdown,
)
from wavedispx.preflight import checks_markdown, plot_preflight_review
from wavedispx.workflow_plotting import (
    plot_workflow_forward,
    plot_workflow_observations,
    plot_workflow_dispersion,
)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('请从 PassiveFW_versionstable 项目内运行本 Notebook')
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f'Project: {PROJECT_ROOT}')


Project: /Volumes/lx_exFAT/PassiveFW_versionstable


## Steps

### 2. 输入参数

只修改下面一个 Cell；模型、观测系统、Shot/子波、频散、拾取和性能参数都集中在这里。

主动源计算关系为

$$w(t)=\left[1-2(\pi f_0t)^2\right]e^{-(\pi f_0t)^2},\qquad u_i(r,\omega)=G_{is}(r,\omega)W(\omega).$$

默认 Hankel/FJ 成像在候选波数 $k=2\pi f/c$ 上对径向炮集积分；理论 Rayleigh/Love 曲线由同一 $V_P,V_S,\rho$ 模型独立求解，只用于校验和叠加，不参与制造频散能量。

- `input_toml=None`：使用本 Cell；设为路径后整个算例由 TOML 接管。
- `shot_csv` 可读取多炮，但一次算例只由 `shot_id` 选择一炮。
- `RECEIVER_LAYOUT` 可切换 `line/grid/file`；文件例子已放在同一 Cell。


In [2]:
# ==================== 只修改这个参数总控 Cell ====================
INPUT_TOML = None  # 例：'examples/active_two_layer.toml'；非 None 时忽略下面参数
RECEIVER_LAYOUT = 'line'  # 'line' / 'grid' / 'file'
SHOT_CSV = None  # 例：'examples/observation_system/shots.csv'
SHOT_ID = None   # SHOT_CSV 多于一炮时必填，例如 'S001'

RECEIVER_PRESETS = {
    'line': {
        'layout': 'line', 'start_x_m': 5.0, 'stop_x_m': 80.0,
        'spacing_m': 5.0, 'y_m': 0.0,
    },
    'grid': {
        'layout': 'grid', 'start_x_m': -25.0, 'stop_x_m': 25.0,
        'spacing_x_m': 5.0, 'start_y_m': -25.0, 'stop_y_m': 25.0,
        'spacing_y_m': 5.0, 'distance_bin_m': 5.0,
    },
    'file': {
        'layout': 'file',
        'coordinates_file': 'examples/observation_system/receivers.csv',
        'distance_bin_m': 5.0,
    },
}

PARAMETERS = {
    'input_toml': INPUT_TOML,
    'template': 'examples/active_quick.toml',
    'run': {
        'name': 'notebook_active_quickstart',
        'output_directory': 'output_python/notebook_active_quickstart',
        'workers': 0, 'frequency_block_size': 8, 'source_block_size': 1,
        'radial_interpolation_step_m': 0.5,
        'store_source_responses': False,
    },
    # 最后一层 thickness_m 必须是 inf（半空间）；Vp 必须大于 Vs
    'model_layers': [
        {'thickness_m': 10.0, 'vp_mps': 380.0, 'vs_mps': 220.0,
         'density_kgm3': 2000.0, 'qp': float('inf'), 'qs': float('inf')},
        {'thickness_m': float('inf'), 'vp_mps': 760.0, 'vs_mps': 440.0,
         'density_kgm3': 2000.0, 'qp': float('inf'), 'qs': float('inf')},
    ],
    'receivers': RECEIVER_PRESETS[RECEIVER_LAYOUT],
    'shot_csv': SHOT_CSV,
    'shot_id': SHOT_ID,
    'active_source': {
        'x_m': 0.0, 'y_m': 0.0, 'component': 'both', 'wavelet': 'ricker',
        'dominant_frequency_hz': 8.0, 'maximum_frequency_hz': 20.0,
        'dt_s': 0.005,
    },
    'processing': {
        'correlation_normalization': 'none',
        'dispersion_method': 'hankel', 'compare_methods': False,
        'frequency_min_hz': 3.0, 'frequency_max_hz': 18.0,
        'velocity_min_mps': 120.0, 'velocity_max_mps': 500.0,
        'velocity_step_mps': 2.0, 'distance_taper_fraction': 0.0,
        'zero_padding': 5.0, 'radial_kernel_lookup_step': 0.002,
        'picker_smoothness': 0.3, 'picker_maximum_jump_mps': 30.0,
        'picker_minimum_energy': 0.05,
        'rayleigh_modes': 3, 'love_modes': 3,
    },
}
# ================================================================


### 3. 构建算例并绘图确认（这里只计算理论曲线，不运行波场正演）


In [3]:
try:
    prepared = prepare_notebook_case(PROJECT_ROOT, 'active', PARAMETERS)
except (OSError, ValueError) as error:
    display(Markdown(input_error_markdown(error)))
    raise
workflow, config, invalidated = prepared.workflow, prepared.config, prepared.invalidated
print(f'输入来源：{prepared.input_origin}')
display(Markdown(parameter_reference_markdown('active')))

review = workflow.review_inputs()
display(Markdown(checks_markdown(review)))
display(Markdown(required_failure_markdown(review)))
figure = plot_preflight_review(review)
display(figure)
plt.close(figure)
assert review.ready, 'required 检查失败：按上面的参数位置和“怎么改”修正总控 Cell'


输入来源：参数 Cell + template: /Volumes/lx_exFAT/PassiveFW_versionstable/examples/active_quick.toml


### 参数详细说明

| 参数路径 | 单位 | 必填/约束 | 作用与调整影响 |
|---|---:|---|---|
| `run.name` | — | 非空 | 算例名；写入图题、manifest 和输出 TOML。 |
| `run.output_directory` | 路径 | 可写目录 | HDF5、CSV、图和配置的输出目录。 |
| `run.workers` | 线程 | 0=自动，≥0 | CPU 并行线程数；正式计算通常用 0。 |
| `run.frequency_block_size` | 频点 | 整数≥1 | C++ 每批频率数；大值更快但更占内存。 |
| `run.source_block_size` | 源 | 整数≥1 | 被动源分块；主动模式影响很小。 |
| `run.radial_interpolation_step_m` | m | >0，且≤最短波长5% | 径向 Green 插值精度；过大会被 required 检查阻止。 |
| `run.store_source_responses` | — | true/false | 是否保存逐噪声源传递函数；通常关闭以节省内存。 |
| `model_layers[].thickness_m` | m | 有限层>0；末层=inf | 层厚；最后一层必须是半空间。 |
| `model_layers[].vp_mps` | m/s | Vp>Vs>0 | P 波速度；影响阻抗与 Rayleigh 椭圆极化。 |
| `model_layers[].vs_mps` | m/s | >0 | S 波速度；控制面波相速度和最短空间波长。 |
| `model_layers[].density_kgm3` | kg/m³ | >0 | 密度；进入层状介质刚度和阻抗。 |
| `model_layers[].qp / qs` | — | >0 或 inf | 品质因子；inf 表示无本征衰减。 |
| `receivers.layout` | — | line/grid/file | 线阵、规则二维面阵或任意二维 CSV。 |
| `receivers.start/stop/spacing` | m | 跨度可被间距整除 | line 使用 x 与 spacing_m；grid 分别使用 x/y。 |
| `receivers.coordinates_file` | CSV | file 时必填 | 表头 x_m,y_m，至少两个不重复有限坐标。 |
| `receivers.distance_bin_m` | m | 被动 grid/file 必填 | 二维台站对转为径向 NCF 道集时的距离箱宽。 |
| `processing.dispersion_method` | — | phase_shift/bessel/hankel | 频散成像核；被动默认 phase_shift，主动默认 hankel。 |
| `processing.compare_methods` | — | true/false | 同时输出三种成像核的同网格对照；默认关闭以节省时间。 |
| `processing.frequency_min/max_hz` | Hz | 0≤fmin<fmax<Nyquist | 频散显示与拾取频带；被动上限不能超过 PSD 上限。 |
| `processing.velocity_min/max_mps` | m/s | 0<vmin<vmax | 相速度扫描范围，必须覆盖预期模态。 |
| `processing.velocity_step_mps` | m/s | >0 | 速度网格步长；更小更细但计算和内存增加。 |
| `processing.distance_taper_fraction` | — | 0–1 | 距离域 Tukey taper；减弱有限孔径截断旁瓣。 |
| `processing.correlation_normalization` | — | spectral_whitening/one_bit/none | NCF 归一化；主动通常 none，被动默认 whitening。 |
| `processing.zero_padding` | 倍 | >0 | 时间 FFT 补零，只细化频率采样，不提高真实分辨率。 |
| `processing.radial_kernel_lookup_step` | — | ≥0 | Bessel/Hankel 核查表步长；0 为逐点精确函数。 |
| `processing.picker_smoothness` | — | ≥0 | 自动拾取连续性惩罚；过大可能压制真实快速变化。 |
| `processing.picker_maximum_jump_mps` | m/s | >0 | 相邻频点允许的最大拾取跳变。 |
| `processing.picker_minimum_energy` | — | 0–1 | 低于该归一化能量的频点不输出拾取。 |
| `processing.rayleigh_modes/love_modes` | 阶 | 整数≥1 | 独立理论频散计算和叠加的模态数。 |
| `shot_csv / shot_id` | CSV / — | 可选；多炮必须选ID | 从 shot_id,x_m,y_m 文件选择一炮；设置后覆盖手填 x/y。 |
| `active_source.x_m / y_m` | m | 有限数值 | 单炮坐标；与接收器的平面距离进入 Green 响应。 |
| `active_source.component` | — | vertical/horizontal/both | Rayleigh、Love 或同时计算。 |
| `active_source.wavelet` | — | 当前为 ricker | 主动震源时间函数。 |
| `active_source.dominant_frequency_hz` | Hz | >0 | Ricker 主频，控制主要能量中心。 |
| `active_source.maximum_frequency_hz` | Hz | ≥主频且<Nyquist | 波场正演频率上限。 |
| `active_source.dt_s` | s | >0 | 时间采样；Nyquist=1/(2dt)，最短周期建议≥8点。 |

| Status | Check | Value | Criterion | Parameter | How to fix |
|---|---|---:|---|---|---|
| ✅ PASS | Temporal Nyquist | 100.00 Hz | > source cutoff 20.00 Hz | active_source.dt_s / maximum_frequency_hz | 减小 dt，或降低源/噪声最高频率，使最高频率严格小于 1/(2dt)。 |
| ✅ PASS | Samples per shortest period | 10.0 | >= 8 recommended | active_source.dt_s / maximum_frequency_hz | 建议减小 dt，使最高频率对应的最短周期至少有 8 个采样点。 |
| ✅ PASS | Spatial alias estimate | 20.53 Hz | >= processing maximum 18.00 Hz | geometry.receivers spacing / processing.frequency_max_hz | 减小接收点间距或径向距离箱，或降低处理最高频率，直到空间混叠上限覆盖处理频带。 |
| ⚠️ WARN | Aperture at minimum frequency | 0.61 wavelengths | >= 2 wavelengths recommended | geometry.receivers aperture / processing.frequency_min_hz | 建议扩大阵列孔径，或提高处理最低频率，使孔径至少覆盖约 2 个波长。 |
| ✅ PASS | Minimum source-receiver distance | 5.00 m | > 0 m | active_source.x_m,y_m / geometry.receivers | 移动与接收器重合的源或炮点；源—接收器最小距离必须大于 0 m。 |

✅ **所有 required 检查通过，可以进入正演。**

<Figure size 1700x1000 with 6 Axes>

## Checks

### 4. 可选分步运行

把相应开关改为 `True`，可在一键运行前查看中间炮集或频散。


In [4]:
RUN_FORWARD_STAGE = False
if RUN_FORWARD_STAGE:
    workflow.compute_forward()
    figure = plot_workflow_forward(workflow)
    display(figure)
    plt.close(figure)
    display(workflow.status())


In [5]:
RUN_DISPERSION_STAGE = False
if RUN_DISPERSION_STAGE:
    workflow.compute_dispersion()
    figure = plot_workflow_dispersion(workflow)
    display(figure)
    plt.close(figure)
    display(workflow.status())


## Results

### 5. 一键全流程

确认输入图后，只改下面第一行为 `True`，运行这一个 Cell 即可正演、成像、拾取、保存并显示完整汇总图。


In [6]:
RUN_ALL_NOW = False

if RUN_ALL_NOW:
    artifacts = workflow.run_all(save=True)
    print(f'完成：{artifacts.output_directory}')
    display(Image(filename=str(artifacts.summary_png)))
else:
    print('尚未启动正演。确认输入后把 RUN_ALL_NOW 改为 True。')


尚未启动正演。确认输入后把 RUN_ALL_NOW 改为 True。


## Next Steps

正式高频算例可直接把 `INPUT_TOML` 指向 `examples/active_two_layer.toml`；高阶多分量和三方法对照读取 `examples/active_high_modes_3c.toml`；日常文件导入和多参数编辑推荐运行 `wavedispx-ui`。
